MODEL

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# ============================================================
# 1. LOAD DATA
# ============================================================
df = pd.read_csv('global_ai_jobs.csv')
print("Data loaded:", df.shape)

# ============================================================
# 2. PREPROCESSING
# ============================================================
# fitur yang relevan berdasarkan EDA
features = ['experience_level', 'experience_years', 'job_role', 'country']
target   = 'salary_usd'

df_model = df[features + [target]].copy()

# Label Encoding untuk experience_level (ada urutan)
experience_order = {'Entry': 1, 'Mid': 2, 'Senior': 3, 'Lead': 4}
df_model['experience_level'] = df_model['experience_level'].map(experience_order)

# One-Hot Encoding untuk country dan job_role (tidak ada urutan)
df_model = pd.get_dummies(df_model, columns=['country', 'job_role'])

print("Shape setelah preprocessing:", df_model.shape)

# ============================================================
# 3. SPLIT DATA (80% training, 20% testing)
# ============================================================
X = df_model.drop(target, axis=1)
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set : {X_train.shape}")
print(f"Testing set  : {X_test.shape}")

# ============================================================
# 4. BANGUN DAN BANDINGKAN MODEL
# ============================================================

# Model 1: Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# Model 2: Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

# Perbandingan hasil
print()
print("=" * 45)
print(f"{'Model':<25} {'MAE':>8} {'R2':>8}")
print("=" * 45)
print(f"{'Linear Regression':<25} ${mean_absolute_error(y_test, lr_pred):>7,.0f} {r2_score(y_test, lr_pred):>8.4f}")
print(f"{'Random Forest':<25} ${mean_absolute_error(y_test, rf_pred):>7,.0f} {r2_score(y_test, rf_pred):>8.4f}")
print("=" * 45)

# ============================================================
# 5. FEATURE IMPORTANCE (Random Forest)
# ============================================================
print()
print("=== Top 10 Fitur Terpenting ===")
importance = pd.Series(rf.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False).head(10).round(4).to_string())

# ============================================================
# 6. SIMPAN MODEL
# ============================================================
joblib.dump(rf, 'salary_model.pkl')
joblib.dump(list(X.columns), 'model_columns.pkl')
print()
print("Model disimpan: salary_model.pkl")
print("Kolom disimpan: model_columns.pkl")

# ============================================================
# 7. CONTOH PREDIKSI
# ============================================================
print()
print("=== Contoh Prediksi ===")

# Buat satu baris data baru
sample = pd.DataFrame([{
    'experience_level': 3,        # Senior
    'experience_years': 6,
    'country_Australia': False, 'country_Brazil': False, 'country_Canada': False,
    'country_France': False, 'country_Germany': False, 'country_India': False,
    'country_Japan': False, 'country_Netherlands': False, 'country_Singapore': False,
    'country_UAE': False, 'country_UK': False, 'country_USA': True,
    'job_role_AI Engineer': False, 'job_role_Computer Vision Engineer': False,
    'job_role_Data Analyst': False, 'job_role_Data Scientist': False,
    'job_role_Machine Learning Engineer': True, 'job_role_NLP Engineer': False,
    'job_role_Research Scientist': False, 'job_role_Software Engineer AI': False
}])

prediksi = rf.predict(sample)[0]
print(f"Profil  : Senior ML Engineer, 6 tahun pengalaman, USA")
print(f"Prediksi gaji: ${prediksi:,.0f}")

Data loaded: (90000, 35)
Shape setelah preprocessing: (90000, 23)
Training set : (72000, 22)
Testing set  : (18000, 22)

Model                          MAE       R2
Linear Regression         $ 11,795   0.8829
Random Forest             $  9,242   0.9280

=== Top 10 Fitur Terpenting ===
experience_years               0.5641
country_India                  0.1527
country_Brazil                 0.1266
job_role_Data Analyst          0.0667
country_USA                    0.0421
country_Singapore              0.0110
job_role_Research Scientist    0.0088
experience_level               0.0067
country_Australia              0.0046
country_Canada                 0.0040

Model disimpan: salary_model.pkl
Kolom disimpan: model_columns.pkl

=== Contoh Prediksi ===
Profil  : Senior ML Engineer, 6 tahun pengalaman, USA
Prediksi gaji: $131,807


EDA

In [ ]:
import pandas as pd

df = pd.read_csv('global_ai_jobs.csv')

# ============================================================
# 1. INFO DASAR DATASET
# ============================================================
print("=== Info Dataset ===")
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print()
print("=== Sample Data ===")
print(df.head(3).to_string())
print()

# ============================================================
# 2. ANALISIS SALARY (TARGET)
# ============================================================
print("=== Statistik Salary ===")
print(f"Min    : ${df['salary_usd'].min():,}")
print(f"Max    : ${df['salary_usd'].max():,}")
print(f"Rata2  : ${df['salary_usd'].mean():,.0f}")
print(f"Median : ${df['salary_usd'].median():,.0f}")
print()

# ============================================================
# 3. PENGARUH EXPERIENCE TERHADAP SALARY
# ============================================================
print("=== Rata-rata Salary per Experience Level ===")
print(df.groupby('experience_level')['salary_usd'].mean().sort_values(ascending=False).round(0))
print()

print("=== Korelasi Experience Years vs Salary ===")
print(df[['experience_years', 'salary_usd']].corr().round(3))
print()

# ============================================================
# 4. PENGARUH JOB ROLE TERHADAP SALARY
# ============================================================
print("=== Rata-rata Salary per Job Role ===")
print(df.groupby('job_role')['salary_usd'].mean().sort_values(ascending=False).round(0))
print()

# ============================================================
# 5. PENGARUH COUNTRY TERHADAP SALARY
# ============================================================
print("=== Rata-rata Salary per Country ===")
print(df.groupby('country')['salary_usd'].mean().sort_values(ascending=False).round(0))
print()

# ============================================================
# 6. FITUR YANG TIDAK BERPENGARUH SIGNIFIKAN
# ============================================================
print("=== Korelasi AI Adoption Score vs Salary ===")
print(df[['ai_adoption_score', 'salary_usd']].corr().round(3))
print()

print("=== Rata-rata Salary per Industry ===")
print(df.groupby('industry')['salary_usd'].mean().sort_values(ascending=False).round(0))
print()

print("=== Rata-rata Salary per Education Required ===")
print(df.groupby('education_required')['salary_usd'].mean().sort_values(ascending=False).round(0))
print()

# ============================================================
# KESIMPULAN EDA
# ============================================================
print("=== Kesimpulan Fitur ===")
print("Berpengaruh    : experience_level, experience_years, job_role, country")
print("Tidak signifikan: ai_adoption_score, industry, education_required")

FileNotFoundError: [Errno 2] No such file or directory: 'global_ai_jobs.csv'